## Running RAG Pipeline

### Installing Libraries

In [ ]:
%pip install voyageai anthropic numpy pypdf

### Building the Index

In [ ]:
from rag_pipeline import RAG

rag = RAG(index_dir="rag_index")   # folder is created next to the notebook
# raw string (r"...") avoids backslash issues
rag.build_index(r".\docs")

### Asking a Question

In [ ]:
text, hits = rag.answer("What does the document say about Diwali festival?")
print(text)

print("\nSources:")
for i, (chunk, score) in enumerate(hits, 1):
    print(f"[{i}] {chunk.source} (chunk {chunk.chunk_id}, relevance {score:.3f})")

### Asking Another Question

In [ ]:
text, hits = rag.answer(
    "Where can I enjoy wildlife safari in India as mentioned in the document?")
print(text)

print("\nSources:")
for i, (chunk, score) in enumerate(hits, 1):
    print(f"[{i}] {chunk.source} (chunk {chunk.chunk_id}, relevance {score:.3f})")

## Running Hybrid RAG Pipeline

### Installing Libraries

In [ ]:
%pip install faiss-cpu rank-bm25 pandas

### Building the Hybrid Index

In [ ]:
from rag_hybrid import HybridRAG

rag = HybridRAG(index_dir="rag_index_hybrid")
rag.build_index(r"./docs", index_type="flat")   # use "hnsw" for large corpora

### Comparing Retrieval Modes

In [ ]:
rag.compare_modes("Which months are the best to visit India?",
                  top_k=5, rerank=False)

### Creating an Evaluation Set

In [ ]:
from rag_eval import generate_eval_set, save_eval_set, load_eval_set

eval_set = generate_eval_set(rag, n=40, path="eval_set.jsonl")
# later sessions: eval_set = load_eval_set("eval_set.jsonl")

### Running the Evaluation

In [ ]:
from rag_eval import evaluate

summary, details = evaluate(rag, eval_set, ks=(1, 3, 5, 10))
summary.round(3)

### Inspecting Failures and Tuning

In [ ]:
# Queries where hybrid+rerank missed entirely in the top 10
from rag_eval import default_configs
misses = details["hybrid+rerank"]
misses[misses["first_hit_rank"].isna()][["query"]]

# Try different fusion weights
configs = {
    "hybrid 1:1": dict(mode="hybrid", rerank=False, candidates=30),
    "hybrid 2:1 vector-heavy": dict(mode="hybrid", rerank=False, candidates=30, vector_weight=2.0),
    "hybrid 1:2 bm25-heavy": dict(mode="hybrid", rerank=False, candidates=30, bm25_weight=2.0),
}
evaluate(rag, eval_set, configs)[0].round(3)